# Full-Study Statistical Analyses

Three statistical layers on top of the panel labels for stronger reviewer-defensible claims:

1. **Confidence intervals for inter-judge κ** — bootstrap CIs for the κ values in
   `inter_judge_agreement_full.json`. Reviewers expect uncertainty bounds on agreement metrics.
2. **Crime-type stratified analysis** — does hallucination behavior vary by crime category?
   Per-crime hallucination rates, plus a chi-square test of independence.
3. **ANOVA / variance partition** — formally tests whether technique or model dominates the
   variance in hallucination rate. Two-way ANOVA on per-triplet hallucination count.

**Inputs:**
- `panel_raw_judge_labels_full.csv` — per-judge labels (for κ and per-triplet metrics)
- `all_triplets_cache.csv` — for crime_type and technique alignment

**Outputs (saved to `C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\`):**
- `kappa_confidence_intervals.json` — bootstrap CIs for all per-pair, per-H κ values
- `crime_type_analysis.csv` and `crime_type_analysis.json` — per-crime hallucination rates + chi-square
- `anova_results.json` — variance partition results
- `statistical_summary.txt` — paper-ready


In [1]:
#pip install statsmodels

In [2]:
import os, json, re
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import cohen_kappa_score

OUTPUT_DIR    = r'C:\Opeyemi\PROMPTS\EVALUATION'
TRIPLETS_PATH = os.path.join(OUTPUT_DIR, 'all_triplets_cache.csv')
PANEL_RAW     = os.path.join(OUTPUT_DIR, 'panel_raw_judge_labels_full.csv')

HALL_DIR = r'C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS'
os.makedirs(HALL_DIR, exist_ok=True)

KAPPA_CI_OUT  = os.path.join(HALL_DIR, 'kappa_confidence_intervals.json')
CRIME_CSV     = os.path.join(HALL_DIR, 'crime_type_analysis.csv')
CRIME_JSON    = os.path.join(HALL_DIR, 'crime_type_analysis.json')
ANOVA_OUT     = os.path.join(HALL_DIR, 'anova_results.json')
SUMMARY_OUT   = os.path.join(HALL_DIR, 'statistical_summary.txt')

HCOLS = ['H1','H2','H3','H4','H5','H6']
N_BOOTSTRAP = 1000
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Load panel labels and triplets
print('Loading panel labels...')
labels = pd.read_csv(PANEL_RAW)
for h in HCOLS:
    labels[h] = pd.to_numeric(labels[h], errors='coerce')
clean = labels[(labels[HCOLS] >= 0).all(axis=1)].copy()
print(f'  Panel rows: {len(clean):,}')

# Per-triplet majority verdict
def majority(col):
    return col.mode().iloc[0] if not col.mode().empty else 0
mv = clean.groupby(['row_idx','model','technique','video','crime_type'])[HCOLS].agg(majority).reset_index()
mv['ANY'] = (mv[HCOLS].sum(axis=1) > 0).astype(int)
mv['hall_count'] = mv[HCOLS].sum(axis=1)
print(f'  Triplets (majority verdicts): {len(mv):,}')


Loading panel labels...
  Panel rows: 19,358
  Triplets (majority verdicts): 9,680


---
## 1. Bootstrap confidence intervals for inter-judge κ

For each pair of judges and each H type, resample the row_idx with replacement and
recompute κ to build a 95% CI. ~1000 bootstrap iterations.

In [3]:
def kappa_bootstrap_ci(y1, y2, n_boot=1000, alpha=0.05, seed=42):
    """Compute bootstrap 95% CI for Cohen's kappa."""
    rng_local = np.random.default_rng(seed)
    n = len(y1)
    y1 = np.asarray(y1, dtype=int)
    y2 = np.asarray(y2, dtype=int)
    boot_kappas = []
    for _ in range(n_boot):
        idx = rng_local.integers(0, n, size=n)
        try:
            k = cohen_kappa_score(y1[idx], y2[idx])
            if not np.isnan(k):
                boot_kappas.append(k)
        except Exception:
            pass
    boot_kappas = np.array(boot_kappas)
    if len(boot_kappas) < 100:
        return None, None, None
    lo = float(np.percentile(boot_kappas, 100 * alpha / 2))
    hi = float(np.percentile(boot_kappas, 100 * (1 - alpha / 2)))
    return lo, hi, float(boot_kappas.std())

# Build pivot: row_idx -> {judge: H1...H6}
pivot = clean.pivot_table(index='row_idx', columns='judge', values=HCOLS)

PAIRS = [('GPT', 'Gemini'), ('GPT', 'Claude'), ('Gemini', 'Claude')]

ci_results = {}
print('Computing bootstrap CIs (1000 iterations per pair × per H)...')
for j1, j2 in PAIRS:
    pair_key = f'{j1}_vs_{j2}'
    print(f'\n  {pair_key}:')
    per_dim = {}
    
    # Per-H
    for h in HCOLS:
        if (h, j1) not in pivot.columns or (h, j2) not in pivot.columns:
            continue
        sub = pivot[[(h, j1), (h, j2)]].dropna()
        if len(sub) < 50:
            per_dim[h] = {'note': 'too few rows'}
            continue
        y1 = sub[(h, j1)].astype(int).values
        y2 = sub[(h, j2)].astype(int).values
        try:
            point = float(cohen_kappa_score(y1, y2))
        except Exception:
            point = None
        if point is None:
            per_dim[h] = {'note': 'kappa undefined'}
            continue
        lo, hi, se = kappa_bootstrap_ci(y1, y2, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED)
        per_dim[h] = {
            'kappa': round(point, 4),
            'ci_low_95': round(lo, 4) if lo is not None else None,
            'ci_high_95': round(hi, 4) if hi is not None else None,
            'bootstrap_se': round(se, 4) if se is not None else None,
            'n': int(len(sub)),
        }
        print(f'    {h}: κ = {point:.3f}  [{lo:.3f}, {hi:.3f}]  n={len(sub):,}')
    
    # Overall (concatenated across all 6 dimensions)
    all_y1, all_y2 = [], []
    for h in HCOLS:
        if (h, j1) not in pivot.columns or (h, j2) not in pivot.columns:
            continue
        sub = pivot[[(h, j1), (h, j2)]].dropna()
        all_y1.extend(sub[(h, j1)].astype(int).values)
        all_y2.extend(sub[(h, j2)].astype(int).values)
    if len(all_y1) > 100:
        try:
            point = float(cohen_kappa_score(all_y1, all_y2))
        except Exception:
            point = None
        if point is not None:
            lo, hi, se = kappa_bootstrap_ci(all_y1, all_y2, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED)
            per_dim['overall'] = {
                'kappa': round(point, 4),
                'ci_low_95': round(lo, 4) if lo is not None else None,
                'ci_high_95': round(hi, 4) if hi is not None else None,
                'bootstrap_se': round(se, 4) if se is not None else None,
                'n': int(len(all_y1)),
            }
            print(f'    overall: κ = {point:.3f}  [{lo:.3f}, {hi:.3f}]  n={len(all_y1):,}')
    
    ci_results[pair_key] = per_dim

with open(KAPPA_CI_OUT, 'w') as f:
    json.dump(ci_results, f, indent=2)
print(f'\nSaved: {KAPPA_CI_OUT}')


Computing bootstrap CIs (1000 iterations per pair × per H)...

  GPT_vs_Gemini:
    H1: κ = 0.774  [0.734, 0.813]  n=3,224
    H2: κ = 0.735  [0.709, 0.761]  n=3,224
    H3: κ = 0.816  [0.795, 0.836]  n=3,224
    H4: κ = 0.605  [0.569, 0.641]  n=3,224
    H5: κ = 0.745  [0.705, 0.785]  n=3,224
    H6: κ = 0.706  [0.681, 0.730]  n=3,224
    overall: κ = 0.817  [0.809, 0.825]  n=19,344

  GPT_vs_Claude:
    H1: κ = 0.106  [0.088, 0.123]  n=3,227
    H2: κ = 0.565  [0.532, 0.595]  n=3,227
    H3: κ = 0.658  [0.632, 0.684]  n=3,227
    H4: κ = 0.624  [0.597, 0.652]  n=3,227
    H5: κ = 0.088  [0.070, 0.106]  n=3,227
    H6: κ = 0.532  [0.503, 0.562]  n=3,227
    overall: κ = 0.502  [0.490, 0.514]  n=19,362

  Gemini_vs_Claude:
    H1: κ = 0.231  [0.206, 0.256]  n=3,227
    H2: κ = 0.691  [0.654, 0.724]  n=3,227
    H3: κ = 0.791  [0.770, 0.811]  n=3,227
    H4: κ = 0.483  [0.455, 0.512]  n=3,227
    H5: κ = 0.343  [0.317, 0.369]  n=3,227
    H6: κ = 0.489  [0.435, 0.534]  n=3,227
    overa

---
## 2. Crime-type stratified analysis

For each crime type (Robbery, Assault, etc.), compute hallucination rates by H type.
Then chi-square tests whether crime_type is independent of "any hallucination."

In [4]:
# Per-crime per-H positive rates
crime_rows = []
for crime in mv['crime_type'].unique():
    sub = mv[mv['crime_type'] == crime]
    row = {'crime_type': crime, 'n_triplets': len(sub)}
    for h in HCOLS:
        row[f'{h}_rate'] = round(sub[h].mean(), 4)
        row[f'{h}_count'] = int(sub[h].sum())
    row['ANY_rate'] = round(sub['ANY'].mean(), 4)
    row['hall_per_triplet'] = round(sub['hall_count'].mean(), 4)
    crime_rows.append(row)

crime_df = pd.DataFrame(crime_rows).sort_values('hall_per_triplet', ascending=False)
crime_df.to_csv(CRIME_CSV, index=False)
print(f'Saved: {CRIME_CSV}\n')

print('=== HALLUCINATION RATE BY CRIME TYPE ===')
print(f'{"Crime":<18}{"n":>6}{"H1":>7}{"H2":>7}{"H3":>7}{"H4":>7}{"H5":>7}{"H6":>7}'
      f'{"ANY":>7}{"halls/T":>10}')
print('-' * 80)
for _, r in crime_df.iterrows():
    print(f'{r["crime_type"]:<18}{r["n_triplets"]:>6,}', end='')
    for h in HCOLS:
        print(f'{r[f"{h}_rate"]*100:>6.1f}%', end='')
    print(f'{r["ANY_rate"]*100:>6.1f}%{r["hall_per_triplet"]:>10.3f}')

# Chi-square: is ANY hallucination independent of crime_type?
print('\n=== CHI-SQUARE: ANY-hallucination by crime_type ===')
ct = pd.crosstab(mv['crime_type'], mv['ANY'])
chi2, p, dof, expected = stats.chi2_contingency(ct)
print(f'chi2 = {chi2:.3f}  p = {p:.4g}  dof = {dof}')
if p < 0.001:
    print('  -> Highly significant: hallucination rate VARIES by crime type.')
elif p < 0.05:
    print('  -> Significant: crime type and hallucination rate are not independent.')
else:
    print('  -> Not significant: hallucination rate is independent of crime type.')

# Also: per-H chi-square
print('\nPer-H chi-square (crime_type independence):')
crime_chi = {}
for h in HCOLS:
    ct_h = pd.crosstab(mv['crime_type'], mv[h])
    if ct_h.shape[1] < 2:
        crime_chi[h] = {'note': 'no variance'}
        continue
    chi2_h, p_h, dof_h, _ = stats.chi2_contingency(ct_h)
    crime_chi[h] = {'chi2': float(chi2_h), 'p': float(p_h), 'dof': int(dof_h)}
    sig = '***' if p_h < 0.001 else ('**' if p_h < 0.01 else ('*' if p_h < 0.05 else ''))
    print(f'  {h}: chi2={chi2_h:.2f}  p={p_h:.4g}  dof={dof_h}  {sig}')

crime_results = {
    'per_crime_rates': {row['crime_type']: row for row in crime_rows},
    'overall_chi2': {'chi2': float(chi2), 'p': float(p), 'dof': int(dof)},
    'per_H_chi2': crime_chi,
}
with open(CRIME_JSON, 'w') as f:
    json.dump(crime_results, f, indent=2, default=str)
print(f'\nSaved: {CRIME_JSON}')


Saved: C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\crime_type_analysis.csv

=== HALLUCINATION RATE BY CRIME TYPE ===
Crime                  n     H1     H2     H3     H4     H5     H6    ANY   halls/T
--------------------------------------------------------------------------------
Shooting             600  64.3%  42.0%  55.8%  40.5%  68.5%  36.8%  92.5%     3.080
Fighting             599  60.1%  23.5%  44.7%  30.7%  64.3%  33.6%  90.1%     2.569
RoadAccidents      1,776  60.0%  17.1%  44.2%  24.0%  66.8%  27.5%  87.4%     2.396
Abuse                599  51.2%  20.0%  45.1%  35.6%  60.9%  23.5%  87.3%     2.364
Vandalism            588  49.7%  30.9%  62.1%  25.7%  50.0%  14.0%  88.9%     2.323
Robbery            1,788  52.9%  16.2%  36.3%  28.6%  62.3%  26.9%  87.8%     2.232
Assault              132  48.5%  17.4%  22.7%  24.2%  64.4%  35.6%  87.9%     2.129
Explosion            598  63.4%  16.7%  22.6%  11.4%  67.4%  29.8%  82.1%     2.112
Shoplifting          600  42.8%  17.2%  36.8%  18

---
## 3. ANOVA — variance partitioning by model vs technique

Two-way ANOVA on per-triplet hallucination count: how much of the variance comes from
the model, the technique, and their interaction? Whichever has the larger F statistic
and effect size dominates.

In [5]:
try:
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    HAVE_STATSMODELS = True
except ImportError:
    print('statsmodels not installed. Falling back to scipy alternatives.')
    print('  For full ANOVA, run: pip install statsmodels')
    HAVE_STATSMODELS = False

# Use hall_count as the dependent variable
df_anova = mv[['model','technique','hall_count','ANY']].copy()

print(f'ANOVA inputs: {len(df_anova):,} rows')
print('Model x technique cell counts:')
print(df_anova.groupby(['model','technique']).size().unstack(fill_value=0))

anova_results = {}

if HAVE_STATSMODELS:
    # Two-way ANOVA on hall_count with interaction term
    print('\n=== Two-way ANOVA: hall_count ~ model * technique ===')
    model_lm = ols('hall_count ~ C(model) + C(technique) + C(model):C(technique)',
                    data=df_anova).fit()
    table = sm.stats.anova_lm(model_lm, typ=2)
    print(table.to_string())
    
    # Convert to JSON-serializable
    anova_count = {}
    for idx in table.index:
        row = table.loc[idx]
        anova_count[idx] = {
            'sum_sq': float(row['sum_sq']),
            'df': float(row['df']),
            'F': float(row['F']) if not pd.isna(row['F']) else None,
            'p': float(row['PR(>F)']) if not pd.isna(row['PR(>F)']) else None,
        }
    
    # Compute eta-squared (effect size = SS_factor / SS_total)
    ss_total = table['sum_sq'].sum()
    eta_squared = {}
    for idx in table.index:
        if idx == 'Residual':
            continue
        eta_squared[idx] = float(table.loc[idx, 'sum_sq'] / ss_total)
    
    print('\nEffect sizes (eta-squared = proportion of total variance explained):')
    for k, v in eta_squared.items():
        print(f'  {k}: η² = {v:.4f}  ({v*100:.2f}% of total variance)')
    
    anova_results['hall_count_two_way'] = {
        'anova_table': anova_count,
        'eta_squared': eta_squared,
        'interpretation': 'Larger eta-squared indicates the factor explains more variance.',
    }
    
    # Same for ANY (binary outcome — logistic but use OLS as approximation here)
    print('\n=== Two-way ANOVA: ANY ~ model * technique ===')
    model_any = ols('ANY ~ C(model) + C(technique) + C(model):C(technique)',
                     data=df_anova).fit()
    table_any = sm.stats.anova_lm(model_any, typ=2)
    print(table_any.to_string())
    
    anova_any = {}
    for idx in table_any.index:
        row = table_any.loc[idx]
        anova_any[idx] = {
            'sum_sq': float(row['sum_sq']),
            'df': float(row['df']),
            'F': float(row['F']) if not pd.isna(row['F']) else None,
            'p': float(row['PR(>F)']) if not pd.isna(row['PR(>F)']) else None,
        }
    ss_total_any = table_any['sum_sq'].sum()
    eta_any = {idx: float(table_any.loc[idx, 'sum_sq'] / ss_total_any)
               for idx in table_any.index if idx != 'Residual'}
    print('\nEffect sizes:')
    for k, v in eta_any.items():
        print(f'  {k}: η² = {v:.4f}  ({v*100:.2f}%)')
    
    anova_results['ANY_two_way'] = {
        'anova_table': anova_any,
        'eta_squared': eta_any,
    }

else:
    # Fallback: scipy one-way ANOVAs
    print('\n=== One-way ANOVA: hall_count by model ===')
    groups = [df_anova[df_anova['model']==m]['hall_count'].values
              for m in df_anova['model'].unique()]
    f_m, p_m = stats.f_oneway(*groups)
    print(f'F = {f_m:.3f}  p = {p_m:.4g}')
    
    print('\n=== One-way ANOVA: hall_count by technique ===')
    groups = [df_anova[df_anova['technique']==t]['hall_count'].values
              for t in df_anova['technique'].unique()]
    f_t, p_t = stats.f_oneway(*groups)
    print(f'F = {f_t:.3f}  p = {p_t:.4g}')
    
    anova_results['fallback'] = {
        'one_way_model': {'F': float(f_m), 'p': float(p_m)},
        'one_way_technique': {'F': float(f_t), 'p': float(p_t)},
        'note': 'statsmodels not installed; full two-way ANOVA skipped',
    }

with open(ANOVA_OUT, 'w') as f:
    json.dump(anova_results, f, indent=2)
print(f'\nSaved: {ANOVA_OUT}')


ANOVA inputs: 9,680 rows
Model x technique cell counts:
technique  Least-to-Most  ReAct  Sequential  Zero-Shot
model                                                 
Claude               807    807         804        807
GPT                  807    807         807        807
Gemini               806    807         807        807

=== Two-way ANOVA: hall_count ~ model * technique ===
                             sum_sq      df           F        PR(>F)
C(model)                3101.979873     2.0  873.224390  0.000000e+00
C(technique)              40.267850     3.0    7.557081  4.786185e-05
C(model):C(technique)    549.067947     6.0   51.521882  9.540682e-63
Residual               17171.955897  9668.0         NaN           NaN

Effect sizes (eta-squared = proportion of total variance explained):
  C(model): η² = 0.1487  (14.87% of total variance)
  C(technique): η² = 0.0019  (0.19% of total variance)
  C(model):C(technique): η² = 0.0263  (2.63% of total variance)

=== Two-way ANOVA: ANY

---
## 4. Paper-ready summary

In [7]:
lines = []
lines.append('=' * 76)
lines.append('FULL-STUDY STATISTICAL ANALYSES — SUMMARY')
lines.append('=' * 76)
lines.append('')

# Kappa CIs
lines.append('-- INTER-JUDGE KAPPA WITH 95% BOOTSTRAP CIs (n=1,000 iterations) --')
lines.append(f'{"Pair":<22}{"H":<6}{"kappa":>8}{"95% CI":>20}{"n":>10}')
lines.append('-' * 66)
for pair, dims in ci_results.items():
    for h, v in dims.items():
        if 'kappa' in v and v.get('ci_low_95') is not None:
            lines.append(f'  {pair:<20}{h:<6}{v["kappa"]:>8.3f}'
                         f'  [{v["ci_low_95"]:.3f}, {v["ci_high_95"]:.3f}]'
                         f'{v["n"]:>10,}')
lines.append('')

# Crime
lines.append('-- HALLUCINATION RATE BY CRIME TYPE --')
lines.append(f'  Overall chi-square: chi2={chi2:.2f}, p={p:.4g}, dof={dof}')
lines.append('  -> ' + ('crime type AFFECTS hallucination rate' if p < 0.05
             else 'crime type does NOT significantly affect hallucination rate'))
lines.append('')
lines.append('  Top 5 highest-hallucination crime types (per triplet):')
top5 = crime_df.head(5)
for _, r in top5.iterrows():
    lines.append(f'    {r["crime_type"]:<18}  ANY={r["ANY_rate"]*100:>5.1f}%  '
                 f'halls/triplet={r["hall_per_triplet"]:.2f}  n={r["n_triplets"]:,}')
lines.append('')
lines.append('  Bottom 3 lowest-hallucination crime types:')
bot3 = crime_df.tail(3)
for _, r in bot3.iterrows():
    lines.append(f'    {r["crime_type"]:<18}  ANY={r["ANY_rate"]*100:>5.1f}%  '
                 f'halls/triplet={r["hall_per_triplet"]:.2f}  n={r["n_triplets"]:,}')

# ANOVA
lines.append('')
lines.append('-- VARIANCE PARTITION (two-way ANOVA on hall_count) --')
if 'hall_count_two_way' in anova_results:
    eta = anova_results['hall_count_two_way']['eta_squared']
    for factor, e in eta.items():
        lines.append(f'  {factor:<32} η² = {e:.4f}  ({e*100:.2f}% of variance)')
    # Identify dominant factor
    dom = max(eta.items(), key=lambda kv: kv[1])
    lines.append(f'  -> Dominant factor: {dom[0]} ({dom[1]*100:.2f}% of variance)')

summary = '\n'.join(lines)
with open(SUMMARY_OUT, 'w', encoding='utf-8') as f:
    f.write(summary)
print(summary)
print(f'\nSaved: {SUMMARY_OUT}')

FULL-STUDY STATISTICAL ANALYSES — SUMMARY

-- INTER-JUDGE KAPPA WITH 95% BOOTSTRAP CIs (n=1,000 iterations) --
Pair                  H        kappa              95% CI         n
------------------------------------------------------------------
  GPT_vs_Gemini       H1       0.774  [0.734, 0.813]     3,224
  GPT_vs_Gemini       H2       0.735  [0.709, 0.761]     3,224
  GPT_vs_Gemini       H3       0.816  [0.795, 0.837]     3,224
  GPT_vs_Gemini       H4       0.605  [0.569, 0.641]     3,224
  GPT_vs_Gemini       H5       0.745  [0.705, 0.785]     3,224
  GPT_vs_Gemini       H6       0.706  [0.681, 0.730]     3,224
  GPT_vs_Gemini       overall   0.817  [0.809, 0.825]    19,344
  GPT_vs_Claude       H1       0.106  [0.088, 0.123]     3,227
  GPT_vs_Claude       H2       0.565  [0.532, 0.596]     3,227
  GPT_vs_Claude       H3       0.658  [0.632, 0.684]     3,227
  GPT_vs_Claude       H4       0.624  [0.597, 0.652]     3,227
  GPT_vs_Claude       H5       0.087  [0.070, 0.106]     3,22